In [1]:
import ast
import os
import optuna
import ipynbname
import pandas as pd
import numpy as np
from optuna.exceptions import TrialPruned
#from Functions.AutoCloud_V4_4 import *
from Functions.AutoCloud_V4_6 import *
from Functions.Utils import *
from Functions.Graphs import *
from Functions.TedaGraphs import *
from Functions.Utils_OPT_R2 import *
from Functions.TedaOptimize_V4_5 import *
import copy

FileName = ipynbname.name()

# ---------------- Dados e modelo ----------------
RS_PATH     = r'Dataset\DSIR1n\B1_1.csv'
HI_PATH     = r'Dataset\HI1\Bearing1_1.csv'
PARAMS_PATH = r'Optimization\TedaRNN_Opt_R2_TESTE_B1_1-B1_2_adapt_symmetric_G0P0\mono\Optimization.csv'
EOL = 0.3
W = 1

c:\Users\Usuario\Desktop\Python\Batteries_Prognostics\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
RS_PATH     = r'Dataset\DSIR1n\B1_1.csv'
HI_PATH     = r'Dataset\HI1\Bearing1_1.csv'
RS = pd.read_csv(RS_PATH)
HI = pd.read_csv(HI_PATH)
sig = HI['PC1'].values
dfP = pd.read_csv(PARAMS_PATH).iloc[:,:11]
params = dfP.sort_values(dfP.columns[0], ascending=True).iloc[0, 1:].values

m,nI,nR,nO,mO,N1,N2,N3,τ,act = ParseParams(params)
m = 1.9
X, Y, Z, kX = PrepareData(RS, HI, W, nI, nO, mO)
ref, J, L = RefEoL(sig, EOL, kX) 

teda = AutoCloud(m=m,nI=nI,nR=nR,nO=nO,ηS=[N1,N2,N3],
			  tau=τ,eol=EOL,ref=ref,act=1,st=0,end=0,
			  wtaG=False,wtaP=False,store=True,feedback='symmetric',mO=mO,
			  rls_mode ='log',nf_rls = 3, ff_rls=1, dt_rls = 1e3
			  )

teda.SimGranularRUL(J,L,X,Y,Z)

PlotSimulation(teda=teda,mode='ply',xi=10, yf=100,ftrs=5,dsi_mode='hide')

In [6]:
RS_PATH     = r'Dataset\DSIR1n\B1_1.csv'
HI_PATH     = r'Dataset\HI1\Bearing1_1.csv'
RS = pd.read_csv(RS_PATH)
HI = pd.read_csv(HI_PATH)

X, Y, Z, kX = PrepareData(RS, HI, W, nI, nO, mO)
ref, J, L = RefEoL(sig, EOL, kX, frac=RUL_FRAC) 
teda.restart(ref)
teda.SimGranularRUL(J,L,X,Y,Z)

PlotSimulation(teda=teda,mode='ply',xi=10, yf=100,ftrs=5,dsi_mode='hide')

In [12]:
df.sort_values(dfP.columns[0], ascending=True)

,WAPE_mean,m,nI,nR,nO,mO,N1,N2,N3,TAU,activation,WAPE_RUL_B1_1,WAPE_HI_B1_1,WAPE_RUL_B1_2,WAPE_HI_B1_2
4,0.351316,2.00,24,[30],1,12,2,8,1,13,1,0.353852,0.036374,0.289794,0.022612
3,0.356967,1.75,25,[24],8,20,1,3,7,2,1,0.458935,0.003285,0.243253,0.008462
2,0.378030,1.75,26,[21],5,6,1,4,7,2,0,0.218487,0.005150,0.519005,0.013419
1,0.553032,1.75,31,[3],5,4,1,3,3,9,0,0.667831,0.044658,0.362299,0.031275
9,0.667015,1.75,10,"[25, 50]",5,2,1,6,9,2,1,0.484891,0.035895,0.665919,0.147326
0,0.675194,2.00,10,[35],1,4,5,3,3,4,1,0.537061,0.297244,0.394141,0.121941
8,0.719973,2.00,16,"[38, 45]",3,0,1,2,3,15,0,0.537532,0.012835,0.880282,0.009296
7,1.304924,2.00,13,"[52, 14]",1,9,8,1,1,9,0,0.993467,0.084111,1.405034,0.127235
6,1.406205,2.00,29,"[22, 45]",1,7,8,2,1,11,0,1.639849,0.080946,1.019761,0.071853
5,1.408325,1.75,8,"[53, 3]",9,1,5,2,4,13,1,0.700109,0.432209,0.972989,0.711343


In [27]:
data = [(f'B{b}', pd.read_csv(fr'Dataset\DSIR1n\B{b}.csv'), pd.read_csv(fr'Dataset\HI1\Bearing{b}.csv'))
        for b in ('1_1', '1_2')]
df = OptimizeSeq2(data=data, FileName='TesteSeq2', OptSampler='tpe', n_study=2, n_trials=500,timeout=600,
                  nLS=[1, 2], nOS=[1, 20], mOS=[0, 20], mS=[1.75, 2])


OPTIMIZATION PROGRESS
Layers (nL): 1-2  |  Session: 2/2 (obj1_L1-2_s2)  |  Sampler: TPE  |  Config: TesteSeq2_B1_1-B1_2_restart_symmetric_G0P0
Trials in this study: 93 stored before this run  |  This run: 500/500  |  Elapsed: 00:07:49 / 00:10:00

BEST TRIAL PER nL
nL=1 | obj1_L1-2_s1 #0385 | Score: 0.2345 | Params: {'m': 1.75, 'nI': 26, 'n_layers': 1, 'nR_layer_0': 23, 'nO': 8, 'N1': 1, 'N2': 4, 'N3': 6, 'τ': 1, 'act': 1, 'mO': 20}   <-- running
nL=2 | obj1_L1-2_s2 #0214 | Score: 0.2724 | Params: {'m': 2.0, 'nI': 15, 'n_layers': 2, 'nR_layer_0': 45, 'nR_layer_1': 57, 'nO': 4, 'N1': 1, 'N2': 1, 'N3': 4, 'τ': 15, 'act': 0, 'mO': 0}   <-- running
Current session best: #0526 | Score: 0.2686

LAST 3 TRIALS
Trial #590 [COMPLETE] | Score: 0.9382 | Params: {'m': 2.0, 'nI': 17, 'n_layers': 1, 'nR_layer_0': 36, 'nO': 1, 'N1': 1, 'N2': 1, 'N3': 5, 'τ': 16, 'act': 0, 'mO': 2}
Trial #591 [COMPLETE] | Score: 0.7257 | Params: {'m': 2.0, 'nI': 18, 'n_layers': 1, 'nR_layer_0': 43, 'nO': 1, 'N1': 1, 'N2

In [28]:
PARAMS_PATH = r'Optimization\TesteSeq2_B1_1-B1_2_restart_symmetric_G0P0\mono\Optimization.csv'
RS_PATH     = r'Dataset\DSIR1n\B1_1.csv'
HI_PATH     = r'Dataset\HI1\Bearing1_1.csv'
RS = pd.read_csv(RS_PATH)
HI = pd.read_csv(HI_PATH)
sig = HI['PC1'].values
dfP = pd.read_csv(PARAMS_PATH).iloc[:,:11]
params = dfP.sort_values(dfP.columns[0], ascending=True).iloc[0, 1:].values

m,nI,nR,nO,mO,N1,N2,N3,τ,act = ParseParams(params)
X, Y, Z, kX = PrepareData(RS, HI, W, nI, nO, mO)
ref, J, L = RefEoL(sig, EOL, kX, frac=RUL_FRAC) 

p = LoadRho()

rho_rul = p['rho_rul'] if RHO_RUL is None and 'rho_rul' in p else RHO_RUL
teda = AutoCloud(m=m,nI=nI,nR=nR,nO=nO,ηS=[N1,N2,N3],act=act,tau=τ,
                    eol=EOL,ref=ref,wtaG=WTAG,wtaP=WTAP,feedback=FEEDBACK,mO=mO,
                    rho=p['rho'],lam=LAM,intr_state=INTR_STATE,
                    rho_dyn=RHO_DYN,rho_eta=p['eta'],rho_T=p['T'],rho_beta=p['beta'],rho_min=RHO_MIN,rho_max=RHO_MAX,
                    rls_mode=RLS_MODE,nf_rls=int(p['nf_rls']) if RLS_MODE == 'pol' else None,ff_rls=p['ff_rls'],dt_rls=p['dt_rls'],
                    rul_bounds=RUL_BOUNDS,rul_growth=RUL_GROWTH,rho_rul=rho_rul,rul_eclip=RUL_ECLIP,rul_frac=RUL_FRAC)
teda.SimGranularRUL(J,L,X,Y,Z)

PlotSimulation(teda=teda,mode='ply',xi=10, yf=100,ftrs=5,dsi_mode='hide')

In [29]:
RS_PATH     = r'Dataset\DSIR1n\B1_2.csv'
HI_PATH     = r'Dataset\HI1\Bearing1_2.csv'
RS = pd.read_csv(RS_PATH)
HI = pd.read_csv(HI_PATH)
sig = HI['PC1'].values
X, Y, Z, kX = PrepareData(RS, HI, W, nI, nO, mO)
ref, J, L = RefEoL(sig, EOL, kX, frac=RUL_FRAC) 
teda.restart(ref)
teda.SimGranularRUL(J,L,X,Y,Z)

PlotSimulation(teda=teda,mode='ply',xi=10, yf=100,ftrs=5,dsi_mode='hide')